# Internal Dependencies
<br>  

### References
- [Analyze java package metrics in a graph database](https://joht.github.io/johtizen/data/2023/04/21/java-package-metrics-analysis.html)
- [Calculate metrics](https://101.jqassistant.org/calculate-metrics/index.html)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from neo4j import GraphDatabase

In [2]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j". 
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [3]:
def get_cypher_query_from_file(cypher_file_name : str):
    with open(cypher_file_name) as file:
        return ' '.join(file.readlines())


def query_cypher_to_data_frame(filename : str, limit: int = -1):
    """
    Execute the Cypher query of the given file and returns the result.
    filename : str : The name of the file containing the Cypher query
    limit : int : The optional limit of rows to optimize the query. Default = -1 = no limit
    """
    cypher_query = get_cypher_query_from_file(filename)
    if limit > 0:
        cypher_query = "{query}\nLIMIT {row_limit}".format(query = cypher_query, row_limit = limit)
    records, summary, keys = driver.execute_query(cypher_query)
    return pd.DataFrame([r.values() for r in records], columns=keys)


def query_first_non_empty_cypher_to_data_frame(*filenames : str, limit: int = -1):
    """
    Executes the Cypher queries of the given files and returns the first result that is not empty.
    If all given file names result in empty results, the last (empty) result will be returned.
    By additionally specifying "limit=" the "LIMIT" keyword will appended to query so that only the first results get returned.
    """    
    result=pd.DataFrame()
    for filename in filenames:
        result=query_cypher_to_data_frame(filename, limit)
        if not result.empty:
            return result
    return result

In [4]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [5]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [6]:
# Pandas DataFrame Display Configuration
pd.set_option('display.max_colwidth', 300)

## Artifacts

List the artifacts this notebook is based on. Different sorting variations help finding artifacts by their features and support larger code bases where the list of all artifacts gets too long.

Only the top 30 entries are shown. The whole table can be found in the following CSV report:  
`List_all_Java_artifacts`

In [7]:
artifacts = query_cypher_to_data_frame("../cypher/Internal_Dependencies/List_all_Java_artifacts.cypher")

### Table 1a - Top 30 artifacts with the highest package count

In [8]:
# Sort by number of packages descending
artifacts.sort_values(by=['packages','artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-messaging-5.0.1.jar,57,570,7,2
1,axon-common-5.0.1.jar,13,150,10,0
2,axon-eventsourcing-5.0.1.jar,7,100,3,4
3,axon-modelling-5.0.1.jar,7,93,2,3
4,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
5,axon-server-connector-5.0.1.jar,5,72,1,4
6,axon-test-5.0.1.jar,5,73,1,3
7,axon-update-5.0.1.jar,5,23,1,1
8,axon-conversion-5.0.1.jar,4,30,4,1
9,axon-metrics-micrometer-5.0.1.jar,2,13,0,2


### Table 1b - Top 30 artifacts with the highest type count

In [9]:
# Sort by number of types descending
artifacts.sort_values(by=['types','artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-messaging-5.0.1.jar,57,570,7,2
1,axon-common-5.0.1.jar,13,150,10,0
2,axon-eventsourcing-5.0.1.jar,7,100,3,4
3,axon-modelling-5.0.1.jar,7,93,2,3
4,axon-test-5.0.1.jar,5,73,1,3
5,axon-server-connector-5.0.1.jar,5,72,1,4
6,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
7,axon-conversion-5.0.1.jar,4,30,4,1
8,axon-update-5.0.1.jar,5,23,1,1
9,axon-metrics-micrometer-5.0.1.jar,2,13,0,2


### Table 1c - Top 30 artifacts with the highest number of incoming dependencies

The following table lists the top 30 artifacts that are used the most by other artifacts (highest count of incoming dependencies, highest in-degree).

In [10]:
# Sort by number of incoming dependencies descending
artifacts.sort_values(by=['incomingDependencies','artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-common-5.0.1.jar,13,150,10,0
1,axon-messaging-5.0.1.jar,57,570,7,2
2,axon-conversion-5.0.1.jar,4,30,4,1
3,axon-eventsourcing-5.0.1.jar,7,100,3,4
4,axon-modelling-5.0.1.jar,7,93,2,3
5,axon-server-connector-5.0.1.jar,5,72,1,4
6,axon-test-5.0.1.jar,5,73,1,3
7,axon-update-5.0.1.jar,5,23,1,1
8,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
9,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7


### Table 1d - Top 30 artifacts with the highest number of outgoing dependencies

The following table lists the top 30 artifacts that are depending on the highest number of other artifacts (highest count of outgoing dependencies, highest out-degree).

In [11]:
# Sort by number of outgoing dependencies descending
artifacts.sort_values(by=['outgoingDependencies','artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
1,axon-eventsourcing-5.0.1.jar,7,100,3,4
2,axon-server-connector-5.0.1.jar,5,72,1,4
3,axon-modelling-5.0.1.jar,7,93,2,3
4,axon-test-5.0.1.jar,5,73,1,3
5,axon-messaging-5.0.1.jar,57,570,7,2
6,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
7,axon-tracing-opentelemetry-5.0.1.jar,1,5,0,2
8,axon-conversion-5.0.1.jar,4,30,4,1
9,axon-update-5.0.1.jar,5,23,1,1


### Table 1e - Top 30 artifacts with the lowest package count

In [12]:
# Sort by number of packages ascending
artifacts.sort_values(by=['packages','artifactName'], ascending=[True, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-tracing-opentelemetry-5.0.1.jar,1,5,0,2
1,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
2,axon-conversion-5.0.1.jar,4,30,4,1
3,axon-server-connector-5.0.1.jar,5,72,1,4
4,axon-test-5.0.1.jar,5,73,1,3
5,axon-update-5.0.1.jar,5,23,1,1
6,axon-eventsourcing-5.0.1.jar,7,100,3,4
7,axon-modelling-5.0.1.jar,7,93,2,3
8,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
9,axon-common-5.0.1.jar,13,150,10,0


### Table 1f - Top 30 artifacts with the lowest type count

In [13]:
# Sort by number of types ascending
artifacts.sort_values(by=['types','artifactName'], ascending=[True, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-tracing-opentelemetry-5.0.1.jar,1,5,0,2
1,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
2,axon-update-5.0.1.jar,5,23,1,1
3,axon-conversion-5.0.1.jar,4,30,4,1
4,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
5,axon-server-connector-5.0.1.jar,5,72,1,4
6,axon-test-5.0.1.jar,5,73,1,3
7,axon-modelling-5.0.1.jar,7,93,2,3
8,axon-eventsourcing-5.0.1.jar,7,100,3,4
9,axon-common-5.0.1.jar,13,150,10,0


### Table 1g - Top 30 artifacts with the lowest number of incoming dependencies

The following table lists the top 30 artifacts that are used the least by other artifacts (lowest count of incoming dependencies, lowest in-degree).

In [14]:
# Sort by number of incoming dependencies ascending
artifacts.sort_values(by=['incomingDependencies','artifactName'], ascending=[True, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
1,axon-spring-boot-autoconfigure-5.0.1.jar,7,70,0,7
2,axon-tracing-opentelemetry-5.0.1.jar,1,5,0,2
3,axon-server-connector-5.0.1.jar,5,72,1,4
4,axon-test-5.0.1.jar,5,73,1,3
5,axon-update-5.0.1.jar,5,23,1,1
6,axon-modelling-5.0.1.jar,7,93,2,3
7,axon-eventsourcing-5.0.1.jar,7,100,3,4
8,axon-conversion-5.0.1.jar,4,30,4,1
9,axon-messaging-5.0.1.jar,57,570,7,2


### Table 1h - Top 30 artifacts with the lowest number of outgoing dependencies

The following table lists the top 30 artifacts that are depending on the lowest number of other artifacts (lowest count of outgoing dependencies, lowest out-degree).

In [15]:
# Sort by number of outgoing dependencies ascending
artifacts.sort_values(by=['outgoingDependencies','artifactName'], ascending=[True, True]).reset_index(drop=True).head(30)

,artifactName,packages,types,incomingDependencies,outgoingDependencies
0,axon-common-5.0.1.jar,13,150,10,0
1,axon-conversion-5.0.1.jar,4,30,4,1
2,axon-update-5.0.1.jar,5,23,1,1
3,axon-messaging-5.0.1.jar,57,570,7,2
4,axon-metrics-micrometer-5.0.1.jar,2,13,0,2
5,axon-tracing-opentelemetry-5.0.1.jar,1,5,0,2
6,axon-modelling-5.0.1.jar,7,93,2,3
7,axon-test-5.0.1.jar,5,73,1,3
8,axon-eventsourcing-5.0.1.jar,7,100,3,4
9,axon-server-connector-5.0.1.jar,5,72,1,4


## Cyclic Dependencies

Cyclic dependencies occur when one package uses a class of another package and vice versa. 
These dependencies can lead to problems when one of these packages needs to be changed.

## Table 2a - Cyclic Dependencies Overview

Show the top 40 cyclic dependencies sorted by the most promising to resolve first. This is done by calculating the number of forward dependencies (first cycle participant to second cycle participant) in relation to backward dependencies (second cycle participant back to first cycle participant). The higher this rate (approaching 1), the easier it should be to resolve the cycle by focussing on the few backward dependencies.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies`

**Columns:**
- *artifactName* identifies the artifact of the first participant of the cycle
- *packageName* identifies the package of the first participant of the cycle
- *dependentArtifactName* identifies the artifact of the second participant of the cycle
- *dependentPackageName* identifies the package of the second participant of the cycle
- *forwardToBackwardBalance* is between 0 and 1. High for many forward and few backward dependencies.
- *numberForward* contains the number of dependencies from the first participant of the cycle to the second one
- *numberBackward* contains the number of dependencies from the second participant of the cycle back to the first one
- *someForwardDependencies* lists some forward dependencies in the text format "type1 -> type2"
- *backwardDependencies* lists the backward dependencies in the format "type1 <- type2" that are recommended to get resolved

In [16]:
cyclic_dependencies = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies.cypher")
cyclic_dependencies.head(40)

,artifactName,packageName,dependentArtifactName,dependentPackageName,forwardToBackwardBalance,numberForward,numberBackward,someForwardDependencies,backwardDependencies
0,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,0.959184,48,1,"[HandlerDefinition->MessageStream, InterceptorChainParameterResolverFactory->Context$ResourceKey, InterceptorChainParameterResolverFactory->Message, InterceptorChainParameterResolverFactory->MessageHandlerInterceptorChain, InterceptorChainParameterResolverFactory->MessageStream, AnnotatedHandler...",[SimpleHandlerAttributes->HandlerAttributes]
1,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling,axon-messaging-5.0.1,org.axonframework.messaging.core,0.942857,34,1,"[InterceptingEventSink->MessageDispatchInterceptor, InterceptingEventSink->MessageStream$Single, InterceptingEventSink->MessageStream, InterceptingEventSink->Message, InterceptingEventSink->MessageStream$Empty, EventHandlingComponent->QualifiedName, TerminalEventMessage->MessageType, NoHandlerFo...",[SubscribableEventSource->EventMessage]
2,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,0.933333,29,1,"[SequenceNumberParameterResolverFactory->ParameterResolver, SequenceNumberParameterResolverFactory->AbstractAnnotatedParameterResolverFactory, EventAppenderParameterResolverFactory->ParameterResolver, EventAppenderParameterResolverFactory->ParameterResolverFactory, EventAppenderParameterResolver...",[HandlerTypeResolver->EventHandler]
3,axon-messaging-5.0.1,org.axonframework.messaging.commandhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,0.894737,18,1,"[CommandDispatcherParameterResolverFactoryConfigurationEnhancer->ParameterResolverFactory, MethodCommandHandlerDefinition$MethodCommandHandlingMember->MessageHandlingMember, MethodCommandHandlerDefinition$MethodCommandHandlingMember->WrappedMessageHandlingMember, CommandHandler->MessageHandler, ...",[HandlerTypeResolver->CommandHandler]
4,axon-messaging-5.0.1,org.axonframework.messaging.queryhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,0.875000,15,1,"[QueryResponse->Message, MethodQueryHandlerDefinition$MethodQueryHandlingMember->WrappedMessageHandlingMember, MethodQueryHandlerDefinition$MethodQueryHandlingMember->UnsupportedHandlerException, MethodQueryHandlerDefinition$MethodQueryHandlingMember->MessageHandlingMember, QueryHandlingMember->...",[HandlerTypeResolver->QueryHandler]
5,axon-modelling-5.0.1,org.axonframework.modelling.annotation,axon-modelling-5.0.1,org.axonframework.modelling,0.846154,12,1,"[EntityIdResolverDefinition->EntityIdResolver, InjectEntityParameterResolverFactory->EntityIdResolver, InjectEntityParameterResolverFactory->PropertyBasedEntityIdResolver, InjectEntity->EntityIdResolver, InjectEntityParameterResolver->StateManager, InjectEntityParameterResolver->EntityIdResolver...",[PropertyBasedEntityIdResolver->TargetEntityIdMemberMismatchException]
6,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.streaming.pooled,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,0.666667,15,3,"[PooledStreamingEventProcessorModule->EventProcessorModule$CustomizationPhase, PooledStreamingEventProcessorModule->EventProcessorCustomization, PooledStreamingEventProcessorModule->EventProcessorConfiguration, PooledStreamingEventProcessorModule->EventProcessorModule$EventHandlingPhase, PooledS...","[EventProcessorModule->PooledStreamingEventProcessorConfiguration, EventProcessorModule->PooledStreamingEventProcessorModule, EventProcessingConfigurer->PooledStreamingEventProcessorsConfigurer]"
7,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.subscribing,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,0.666667,15,3,"[SubscribingEventProcessorsConf

### Table 2b - Cyclic Dependencies Break Down

Lists packages with cyclic dependencies with every dependency in a separate row sorted by the most promising  dependency first.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies_Breakdown`

**Columns in addition to Table 2a:**
- *dependency* shows the cycle dependency in the text format "type1 -> type2" (forward) or "type2<-type1" (backward)

In [17]:
cyclic_dependencies_breakdown = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies_Breakdown.cypher",limit=40)
cyclic_dependencies_breakdown

,artifactName,packageName,dependentArtifactName,dependentPackageName,dependency,forwardToBackwardBalance,numberForward,numberBackward
0,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,DefaultParameterResolverFactory$AnnotatedMetadataParameterResolver->Message,0.959184,48,1
1,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,AggregateTypeParameterResolverFactory$AggregateTypeParameterResolver->Context$ResourceKey,0.959184,48,1
2,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,MethodInvokingMessageHandlingMember->MessageStream$Entry,0.959184,48,1
3,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,DefaultParameterResolverFactory$MessageParameterResolver->Message,0.959184,48,1
4,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,DefaultParameterResolverFactory$MetadataParameterResolver->Metadata,0.959184,48,1
5,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,AggregateTypeParameterResolverFactory$AggregateTypeParameterResolver->LegacyResources,0.959184,48,1
6,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,MethodInvokingMessageHandlingMember->MessageStream$Single,0.959184,48,1
7,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,MethodInvokingMessageHandlingMember->DelayedMessageStream,0.959184,48,1
8,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,AnnotatedMessageHandlingMemberDefinition->Message,0.959184,48,1
9,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,SourceIdParameterResolverFactory$SourceIdParameterResolver->Context$ResourceKey,0.959184,48,1


### Table 2c - Cyclic Dependencies Break Down - Backward Dependencies Only

Lists packages with cyclic dependencies with every dependency in a separate row sorted by the most promising  dependency first. This table only contains the backward dependencies from the second participant of the cycle back to the first one that are the most promising to resolve.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies_Breakdown_BackwardOnly`

In [18]:
cyclic_dependencies_breakdown_backward = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies_Breakdown_Backward_Only.cypher",limit=40)
cyclic_dependencies_breakdown_backward

,artifactName,packageName,dependentArtifactName,dependentPackageName,dependency,forwardToBackwardBalance,numberForward,numberBackward
0,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core,HandlerAttributes<-SimpleHandlerAttributes,0.959184,48,1
1,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling,axon-messaging-5.0.1,org.axonframework.messaging.core,EventMessage<-SubscribableEventSource,0.942857,34,1
2,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,EventHandler<-HandlerTypeResolver,0.933333,29,1
3,axon-messaging-5.0.1,org.axonframework.messaging.commandhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,CommandHandler<-HandlerTypeResolver,0.894737,18,1
4,axon-messaging-5.0.1,org.axonframework.messaging.queryhandling.annotation,axon-messaging-5.0.1,org.axonframework.messaging.core.annotation,QueryHandler<-HandlerTypeResolver,0.875000,15,1
5,axon-modelling-5.0.1,org.axonframework.modelling.annotation,axon-modelling-5.0.1,org.axonframework.modelling,TargetEntityIdMemberMismatchException<-PropertyBasedEntityIdResolver,0.846154,12,1
6,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.streaming.pooled,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,PooledStreamingEventProcessorsConfigurer<-EventProcessingConfigurer,0.666667,15,3
7,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.streaming.pooled,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,PooledStreamingEventProcessorModule<-EventProcessorModule,0.666667,15,3
8,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.streaming.pooled,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,PooledStreamingEventProcessorConfiguration<-EventProcessorModule,0.666667,15,3
9,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.processing.subscribing,axon-messaging-5.0.1,org.axonframework.messaging.eventhandling.configuration,SubscribingEventProcessorConfiguration<-EventProcessorModule,0.666667,15,3


## Interface Segregation Candidates

Well known from [Design Principles and Design Patterns by Robert C. Martin](http://staff.cs.utu.fi/~jounsmed/doos_06/material/DesignPrinciplesAndPatterns.pdf), the *Interface Segregation Principle* suggests that software components should have narrow, focused interfaces rather than large, general-purpose ones. The goal is to minimize the dependencies between components and increase modularity, flexibility, and maintainability.

Smaller, focused and purpose-driven interfaces

- make it easier to modify individual components without affecting the rest of the system.
- make it clearer which client is affected by which change.
- don’t force their clients to depend on methods they don’t need.
- reduce the scope of changes since a change to one component doesn’t affect others.
- lead to a more loosely coupled architecture that is easier to understand and maintain.

Reference: [Analyze java package metrics in a graph database](https://joht.github.io/johtizen/data/2023/04/21/java-package-metrics-analysis.html#interface-segregation)

### How to apply the results

If just one method of a type is used, especially in many places, then the result of this method can be used to call e.g. a method or constuct an object instead of using the whole object and then just calling that single method.

If there are a couple of methods that are used for a distinct purpose, those could be factored out into a separate interface. The original type can extended/implement the new interface so that there are no breaking changes. Then all the callers, that use only this group of methods, can be changed to the new interface.


### Table 4 - Top 40 most used combinations of methods

The following table shows the top 40 most used combinations of methods of larger types that might benefit from applying the *Interface Segregation Principle*. The whole table can be found in the CSV report `Candidates_for_Interface_Segregation`.

In [19]:
interface_segregation_candidates=query_cypher_to_data_frame("../cypher/Internal_Dependencies/Candidates_for_Interface_Segregation.cypher", limit=40)
interface_segregation_candidates

,fullDependentTypeName,declaredMethods,calledMethodNames,calledMethods,callerTypes
0,org.axonframework.messaging.core.unitofwork.ProcessingContext,32,[computeResourceIfAbsent],1,7
1,org.axonframework.messaging.core.unitofwork.UnitOfWork,24,[executeWithResult],1,6
2,org.axonframework.common.configuration.ComponentDefinition$ComponentCreator,17,[createComponent],1,5
3,org.axonframework.messaging.commandhandling.CommandBus,6,[dispatch],1,5
4,org.axonframework.messaging.core.unitofwork.ProcessingContext,32,[withResource],1,4
5,org.axonframework.messaging.eventhandling.EventMessage,19,"[identifier, timestamp]",2,4
6,org.axonframework.messaging.eventhandling.EventMessage,19,[timestamp],1,4
7,org.axonframework.messaging.core.MessageStream$Entry,7,[message],1,4
8,org.axonframework.messaging.core.MessageStream$Empty,44,[cast],1,3
9,org.axonframework.messaging.core.DelayedMessageStream,42,[create],1,3


## Package Usage

### Table 5 - Types that are used by multiple packages

This table shows the top 40 packages that are used by the highest number of different packages. The whole table can be found in the CSV report `List_types_that_are_used_by_many_different_packages`.


In [20]:
types_used_by_many_packages=query_cypher_to_data_frame("../cypher/Internal_Dependencies/List_types_that_are_used_by_many_different_packages.cypher", limit=40)
types_used_by_many_packages

,fullQualifiedDependentTypeName,dependentTypeName,dependentTypeLabels,numberOfUsingPackages
0,org.axonframework.messaging.core.unitofwork.ProcessingContext,ProcessingContext,"[Type, File, Java, ByteCode, Interface, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mar...",57
1,org.axonframework.common.annotation.Internal,Internal,"[Type, File, Java, ByteCode, Annotation, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponent...",47
2,org.axonframework.messaging.core.Message,Message,"[Type, File, Java, ByteCode, Interface, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mar...",46
3,org.axonframework.common.infra.ComponentDescriptor,ComponentDescriptor,"[Type, File, Java, ByteCode, Interface, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponent0...",39
4,org.axonframework.messaging.eventhandling.EventMessage,EventMessage,"[Type, File, Java, ByteCode, Interface, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponen...",35
5,org.axonframework.messaging.core.MessageStream,MessageStream,"[Type, File, Java, ByteCode, Interface, GenericDeclaration, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWea...",35
6,org.axonframework.messaging.core.MessageType,MessageType,"[Type, File, Java, ByteCode, Record, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponent0, Mark4TypeLabelPropagation3, M...",29
7,org.axonframework.messaging.core.QualifiedName,QualifiedName,"[Type, File, Java, ByteCode, Record, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityBetweenness, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4T...",29
8,org.axonframework.common.configuration.Configuration,Configuration,"[Type, File, Java, ByteCode, Interface, Mark4TopCentralityPageRank, Mark4TopCentralityArticleRank, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponent0, Mark4TypeLabelPropagation9, Mark4TypeLouvainCommunity0,...",28
9,org.axonframework.common.FutureUtils,FutureUtils,"[Type, File, Java, Class, ByteCode, Mark4TopCentralityHarmonic, Mark4TopCentralityCloseness, Mark4TopCentralityHyperlinkInducedTopicSearchAuthority, Mark4TopCentralityHyperlinkInducedTopicSearchHub, Mark4TypeWeaklyConnectedComponent0, Mark4TypeLabelPropagation14, Mark4TypeLouvainCommunity8, Mark...",26


### Table 6 - Packages that are used by multiple artifacts

This table shows the top 30 artifacts that only use a few (compared to all existing) packages of another artifact.
The whole table can be found in the CSV report `ArtifactPackageUsage`.

In [21]:
used_packages_of_dependent_artifact=query_cypher_to_data_frame("../cypher/Internal_Dependencies/How_many_packages_compared_to_all_existing_are_used_by_dependent_artifacts.cypher",limit=30)
used_packages_of_dependent_artifact

,artifactName,dependentArtifactName,dependentPackages,dependentArtifactPackages,packageUsagePercentage,dependentFullQualifiedPackageNames,dependentPackageNames
0,axon-tracing-opentelemetry-5.0.1,axon-messaging-5.0.1,2,57,0.035088,"[org.axonframework.messaging.tracing, org.axonframework.messaging.core]","[tracing, core]"
1,axon-tracing-opentelemetry-5.0.1,axon-common-5.0.1,1,13,0.076923,[org.axonframework.common],[common]
2,axon-metrics-micrometer-5.0.1,axon-messaging-5.0.1,6,57,0.105263,"[org.axonframework.messaging.queryhandling, org.axonframework.messaging.monitoring, org.axonframework.messaging.core, org.axonframework.messaging.eventhandling, org.axonframework.messaging.eventhandling.processing, org.axonframework.messaging.commandhandling]","[queryhandling, monitoring, core, eventhandling, processing, commandhandling]"
3,axon-test-5.0.1,axon-messaging-5.0.1,8,57,0.140351,"[org.axonframework.messaging.commandhandling, org.axonframework.messaging.core, org.axonframework.messaging.monitoring, org.axonframework.messaging.core.unitofwork, org.axonframework.messaging.eventhandling.processing.streaming.token, org.axonframework.messaging.eventstreaming, org.axonframework...","[commandhandling, core, monitoring, unitofwork, token, eventstreaming, eventhandling, annotation]"
4,axon-server-connector-5.0.1,axon-modelling-5.0.1,1,7,0.142857,[org.axonframework.modelling],[modelling]
5,axon-server-connector-5.0.1,axon-eventsourcing-5.0.1,1,7,0.142857,[org.axonframework.eventsourcing.eventstore],[eventstore]
6,axon-test-5.0.1,axon-eventsourcing-5.0.1,1,7,0.142857,[org.axonframework.eventsourcing.eventstore],[eventstore]
7,axon-metrics-micrometer-5.0.1,axon-common-5.0.1,2,13,0.153846,"[org.axonframework.common.configuration, org.axonframework.common]","[configuration, common]"
8,axon-spring-boot-autoconfigure-5.0.1,axon-server-connector-5.0.1,1,5,0.200000,[org.axonframework.axonserver.connector],[connector]
9,axon-spring-boot-autoconfigure-5.0.1,axon-test-5.0.1,1,5,0.200000,[org.axonframework.test.server],[server]


### Table 7 - Types that are used by multiple artifacts

This table shows the top 30 types that only use a few (compared to all existing) types of another artifact. The whole table can be found in the CSV report `ClassesPerPackageUsageAcrossArtifacts`.

In [22]:
used_types_of_dependent_artifact=query_cypher_to_data_frame("../cypher/Internal_Dependencies/How_many_classes_compared_to_all_existing_in_the_same_package_are_used_by_dependent_packages_across_different_artifacts.cypher", limit=30)
used_types_of_dependent_artifact

,artifactName,dependentArtifactName,packageName,dependentPackage.fqn,dependentTypes,dependentPackageTypes,typeUsagePercentage,dependentTypeNames
0,axon-eventsourcing-5.0.1,axon-messaging-5.0.1,org.axonframework.eventsourcing.configuration,org.axonframework.messaging.core,1,80,0.012500,[org.axonframework.messaging.core.MessageTypeResolver]
1,axon-test-5.0.1,axon-messaging-5.0.1,org.axonframework.test.matchers,org.axonframework.messaging.core,1,80,0.012500,[org.axonframework.messaging.core.Message]
2,axon-modelling-5.0.1,axon-messaging-5.0.1,org.axonframework.modelling.repository,org.axonframework.messaging.core,1,80,0.012500,[org.axonframework.messaging.core.Context$ResourceKey]
3,axon-eventsourcing-5.0.1,axon-messaging-5.0.1,org.axonframework.eventsourcing.configuration,org.axonframework.messaging.core.annotation,1,50,0.020000,[org.axonframework.messaging.core.annotation.ParameterResolverFactory]
4,axon-spring-boot-autoconfigure-5.0.1,axon-messaging-5.0.1,org.axonframework.extension.springboot.autoconfig,org.axonframework.messaging.core.annotation,1,50,0.020000,[org.axonframework.messaging.core.annotation.HandlerEnhancerDefinition]
5,axon-modelling-5.0.1,axon-common-5.0.1,org.axonframework.modelling.entity.annotation,org.axonframework.common.configuration,1,46,0.021739,[org.axonframework.common.configuration.Configuration]
6,axon-eventsourcing-5.0.1,axon-common-5.0.1,org.axonframework.eventsourcing.annotation,org.axonframework.common.configuration,1,46,0.021739,[org.axonframework.common.configuration.Configuration]
7,axon-messaging-5.0.1,axon-common-5.0.1,org.axonframework.messaging.core,org.axonframework.common.configuration,1,46,0.021739,[org.axonframework.common.configuration.Configuration]
8,axon-messaging-5.0.1,axon-common-5.0.1,org.axonframework.messaging.core.unitofwork,org.axonframework.common.configuration,1,46,0.021739,[org.axonframework.common.configuration.ComponentNotFoundException]
9,axon-eventsourcing-5.0.1,axon-common-5.0.1,org.axonframework.eventsourcing.annotation.reflection,org.axonframework.common.configuration,1,46,0.021739,[org.axonframework.common.configuration.Configuration]


### Table 8 - Duplicate package names across artifacts

This table shows the top 30 duplicate package names across artifacts. They are ordered by the number of duplicates descending.

This might lead to confusion, makes importing more error prone and might even lead to duplicate classes where only one of them will be loaded by the class loader. If a package is named the same way in two or more artifacts this even allows another artifact to access package protected classes, methods or members which might not be intended. 

The whole table can be found in the CSV report `DuplicatePackageNamesAcrossArtifacts`.

In [23]:
duplicate_package_names_across_artifacts=query_cypher_to_data_frame("../cypher/Artifact_Dependencies/Artifacts_with_duplicate_packages.cypher", limit=30)
duplicate_package_names_across_artifacts

,packageName,duplicates,artifactNames


### Table 9 - Annotated elements

This table shows 30 most used Java Annotations including some examples where they are used.


In [24]:
annotated_elements=query_cypher_to_data_frame("../cypher/Java/Annotated_code_elements.cypher", limit=30)
annotated_elements

,annotationName,languageElement,numberOfAnnotatedElements,examples
0,jakarta.annotation.Nonnull,Parameter,3382,"[org.axonframework.axonserver.connector.MetadataConverter.convertGrpcToMetadataValues(0), org.axonframework.axonserver.connector.MetadataConverter.convertMetadataValuesToGrpc(0), org.axonframework.axonserver.connector.util.PriorityExecutorService.awaitTermination(1), org.axonframework.axonserver..."
1,jakarta.annotation.Nonnull,Method,833,"[org.axonframework.axonserver.connector.MetadataConverter.convertGrpcToMetadataValues, org.axonframework.axonserver.connector.MetadataConverter.convertMetadataValuesToGrpc, org.axonframework.axonserver.connector.util.PriorityExecutorService.shutdownNow, org.axonframework.axonserver.connector.uti..."
2,jakarta.annotation.Nullable,Parameter,399,"[org.axonframework.axonserver.connector.util.ExceptionConverter.convertToErrorMessage(1), org.axonframework.axonserver.connector.query.AxonServerQueryBusConnector.query(1), org.axonframework.axonserver.connector.query.AxonServerQueryBusConnector.subscriptionQuery(1), org.axonframework.axonserver..."
3,jakarta.annotation.Nullable,Method,99,"[org.axonframework.test.fixture.RecordingCommandBus.resultOf, org.axonframework.modelling.annotation.InjectEntityParameterResolverFactory.createInstance, org.axonframework.modelling.entity.annotation.AnnotatedEntityMetamodel.getExpectedRepresentation, org.axonframework.modelling.entity.child.Com..."
4,org.axonframework.common.annotation.Internal,Class,75,"[org.axonframework.axonserver.connector.MetadataConverter, org.axonframework.axonserver.connector.query.AbstractQueryResponseMessageStream, org.axonframework.axonserver.connector.query.QueryConverter, org.axonframework.axonserver.connector.query.QueryResponseMessageStream, org.axonframework.axon..."
5,jakarta.annotation.Nonnull,Field,70,"[org.axonframework.axonserver.connector.command.AxonServerCommandBusConnector$FutureResultCallback.result, org.axonframework.axonserver.connector.command.AxonServerCommandBusConnector$FutureResultCallback.command, org.axonframework.test.fixture.AxonTestFixture$Customization.fieldFilters, org.axo..."
6,java.lang.FunctionalInterface,Interface,55,"[org.axonframework.axonserver.connector.ErrorCode$ExceptionBuilder, org.axonframework.axonserver.connector.ManagedChannelCustomizer, org.axonframework.axonserver.connector.InstructionAckSource, org.axonframework.axonserver.connector.TargetContextResolver, org.axonframework.axonserver.connector.T..."
7,java.lang.annotation.Retention,Annotation,42,"[org.axonframework.extension.springboot.util.RegisterDefaultEntities, org.axonframework.extension.springboot.util.ConditionalOnQualifiedBean, org.axonframework.extension.springboot.util.ConditionalOnMissingQualifiedBean, org.axonframework.modelling.annotation.TargetEntityId, org.axonframework.mo..."
8,java.lang.annotation.Target,Annotation,42,"[org.axonframework.extension.springboot.util.RegisterDefaultEntities, org.axonframework.extension.springboot.util.ConditionalOnQualifiedBean, org.axonframework.extension.springboot.util.ConditionalOnMissingQualifiedBean, org.axonframework.modelling.annotation.TargetEntityId, org.axonframework.mo..."
9,org.springframework.context.annotation.Bean,Method,36,"[org.axonframework.extension.springboot.autoconfig.AxonTimeoutAutoConfiguration.messageTimeoutHandlerEnhancerDefinition, org.axonframework.extension.springboot.autoconfig.AxonTimeoutAutoConfiguration.axonTimeoutConfigurationEnhancer, org.axonframework.extension.springboot.autoconfig.AvroSchemaSt..."


### Table 10 - Distance distribution between dependent files

This table shows the file directory distance distribution between dependent files. Intuitively, the distance is given by the fewest number of change directory commands needed to navigate between a file and a dependency it uses. Those are aggregate to see how many dependent files are in the same directory, how many are just one change directory command apart, and so on.

In [25]:
query_first_non_empty_cypher_to_data_frame("../cypher/Internal_Dependencies/Get_file_distance_as_shortest_contains_path_for_dependencies.cypher",
                                           "../cypher/Internal_Dependencies/Set_file_distance_as_shortest_contains_path_for_dependencies.cypher", limit=20)

,dependency.fileDistanceAsFewestChangeDirectoryCommands,numberOfDependencies,numberOfDependencyUsers,numberOfDependencyProviders,examples
0,0,2080,854,896,"[/axon-spring-boot-autoconfigure-5.0.1.jar uses /axon-server-connector-5.0.1.jar, /org/axonframework/axonserver/connector/command uses /org/axonframework/axonserver/connector/util, /org/axonframework/axonserver/connector/query uses /org/axonframework/axonserver/connector/util, /org/axonframework..."
1,1,96,82,41,"[/org/axonframework/axonserver/connector/query uses /org/axonframework/axonserver/connector, /org/axonframework/axonserver/connector/command uses /org/axonframework/axonserver/connector, /org/axonframework/axonserver/connector/util uses /org/axonframework/axonserver/connector, /org/axonframework..."
2,2,2067,606,417,"[/org/axonframework/axonserver/connector/command/CommandConverter.class uses /org/axonframework/axonserver/connector/MetadataConverter.class, /org/axonframework/axonserver/connector/event/AggregateBasedAxonServerEventStorageEngine.class uses /org/axonframework/axonserver/connector/MetadataConver..."
3,4,2019,650,309,"[/org/axonframework/extension/springboot uses /org/axonframework/axonserver/connector, /org/axonframework/extension/springboot/actuator/axonserver uses /org/axonframework/axonserver/connector, /org/axonframework/extension/springboot/autoconfig uses /org/axonframework/axonserver/connector, /org/a..."
